In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import os
import time
import numpy as np
from itertools import chain
import pickle
from pathlib import Path
from glob import glob
import traceback
from copy import deepcopy

from utils.write_output import write_array2txt

from cylinder_extraction.station import Station
from cylinder_extraction.cylinder import Cylinder
from cylinder_alignment.station_alignment import StationAlignment
from utils.utilities import cart2pol, pol2cart

In [3]:
# data_dir = 'D:/REGISTRATION DATA/External/JB-448 26ft EXT'
data_dir = Path('D:/REGISTRATION DATA/External/JB-448 26ft EXT')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]

# data_dir = '/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483'
# filename_template = '{}.txt'
# station_names = ['055']
# ptc_col_names =  ["x","y","z"]
# data_dir = 'D:\REGISTRATION DATA\External\JB-483 70ft EXT'
# station_names = ['{:03d}'.format(i) for i in range(55,68)]
# filename_template = 'Jb-483 Scan_SuncorTk20D71Scan_{}.e57'
# ptc_col_names =  ["x","y","z"]


# Read station data

This data were saved after extracting cylinders

In [4]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
for fp in source_files:
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn.split('_')[-1]
    station_names.append(sn)
stations = []
for sn in station_names:
    pkl_fp = os.path.join(
        data_dir,
        f'{sn}-output',
        f'{sn}.pkl'
    )
    with open(pkl_fp, 'rb') as f:
        st = pickle.load(f)
        stations.append(st)


STATION Jb-448 Exports_FARO_LS03550_088
STATION Jb-448 Exports_FARO_LS03550_089
STATION Jb-448 Exports_FARO_LS03550_090
STATION Jb-448 Exports_FARO_LS03550_091
STATION Jb-448 Exports_FARO_LS03550_092
STATION Jb-448 Exports_FARO_LS03550_093
STATION Jb-448 Exports_FARO_LS03550_094
STATION Jb-448 Exports_FARO_LS03550_095
STATION Jb-448 Exports_FARO_LS03550_096


In [ ]:
for s in stations:
    print(f'station {s.name}')
    for c in s.refined_cylinders:
        print(c.radius, end=' ')
    print('\n')

# Alignment

In [8]:
s.refined_cylinders

[{'station': {'name': '096'}, 'radius': 814.7722, 'axis': "{'position': array([446.25137 , 653.6966  ,  34.013397], dtype=float32), 'vector': array([ 5.0937186e-05, -5.2002292e-02,  9.9864697e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-448 26ft EXT\\096-output\\096 cluster_1.txt'},
 {'station': {'name': '096'}, 'radius': 4.865996, 'axis': "{'position': array([-30.127346 ,  20.753937 ,  -0.7406107], dtype=float32), 'vector': array([-0.04163696, -0.02483111,  0.9988242 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-448 26ft EXT\\096-output\\096 cluster_6.txt'},
 {'station': {'name': '096'}, 'radius': 3.988564, 'axis': "{'position': array([-13.1880865 ,   7.415807  ,  -0.29315326], dtype=float32), 'vector': array([-0.03569156, -0.02374425,  0.9990807 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-448 26ft EXT\\096-output\\096 cluster_8.txt'},
 {'station': {'name': '096'}, 'radius': 2705.8582, 'axis': "{

In [6]:
out_dir = os.path.join(data_dir, 'top_center_aligned_with_radius_extended')
if not os.path.exists(out_dir):
    os.mkdir(out_dir)
# reference cylinder
c_ref = stations[0].refined_cylinders[2]
# a patch
ptc_fn = os.path.join(
    data_dir,
    f'{c_ref.station.name}-output',
    c_ref.ptc_filepath.split('\\')[-1]
)
c_ref = Cylinder(
    radius = c_ref.radius,
    position = c_ref.axis.position,
    vector = c_ref.axis.vector,
    metadata = {
        'ptc_filepath': ptc_fn,
        'station': c_ref.station,
        'id': c_ref.id
    }
)
c_ref.read_points()
c_ref.estimate_top_surface()
center_ref = c_ref.top_surface['center']
# center_ref = c_ref.axis.intersect_xy()
for s in stations:
    # if s.name != '057':
    #     continue
    try:
        cand_cyls = [c for c in s.refined_cylinders if c.radius > 9 and c.radius < 11.5]
        # print(len(cand_r))
        if len(cand_cyls) > 1:
            print(cand_cyls)
            # get the one that whose radius is closer to 10.6
            rs = [abs(c.radius-10.67)  for c in cand_cyls]
            i = rs.index(min(rs))
            # i = int(input('specify target: '))
            target_cyl = cand_cyls[i]
        else:
            target_cyl = cand_cyls[0]
        # a patch
        ptc_fn = os.path.join(
            data_dir,
            f'{target_cyl.station.name}-output',
            target_cyl.ptc_filepath.split('\\')[-1]
        )
        ori_target_cyl = deepcopy(target_cyl)

        target_cyl = Cylinder(
            radius = target_cyl.radius,
            position = target_cyl.axis.position,
            vector = target_cyl.axis.vector,
            metadata = {
                'ptc_filepath': ptc_fn,
                'station': target_cyl.station,
                'id': target_cyl.id
            }
        )
        target_cyl.read_points()
        # center
        target_cyl.estimate_top_surface()
        center = target_cyl.top_surface['center']
        # save
        s.target_cylinder = target_cyl
        
        # center = target_cyl.axis.intersect_xy()
        # translation
        t = center_ref - center
        target_cyl.match = [{
            'cylinder_ref': c_ref,
            'translation': t
        }]

        # read points from related station
        _s = Station(
            file_path = target_cyl.station.file_path
        )

        target_cyl.points = _s.ptc

        # extend points
        rho, theta = cart2pol(
            target_cyl.points[:,0] - center[0], 
            target_cyl.points[:,1] - center[1]
        )

        # 1m + radius
        mask = rho <= 4.0 + target_cyl.radius
        target_cyl.points = target_cyl.points[mask,:]
        
        # add big clusters in surrounding environment
        for cyl in s.refined_cylinders:
            if ori_target_cyl == cyl:
                print('skip')
                continue
            cyl.read_points()
            target_cyl.points = np.r_[target_cyl.points, cyl.points]

        # translate and save
        write_array2txt(
            os.path.join(out_dir, f'{s.name}.txt'),
            target_cyl.points + target_cyl.match[0]['translation'].reshape([1,3])
        )
        # write_array2txt(
        #     os.path.join(out_dir, f'{s.name}.txt'),
        #     points + t
            # cyl.ptc + t
        # )
    except Exception as exc:
        print(f'Error in {s}')
        traceback.print_exc()


top grid 53
top grid 53
polar grid cell: 57
Error in {'name': '088'}
Error in {'name': '089'}
Error in {'name': '090'}
Error in {'name': '091'}
Error in {'name': '092'}
Error in {'name': '093'}
Error in {'name': '094'}
Error in {'name': '095'}
Error in {'name': '096'}


Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent call last):
  File "<ipython-input-6-713df3781530>", line 40, in <module>
    target_cyl = cand_cyls[0]
IndexError: list index out of range
Traceback (most recent

In [48]:
stations[3].cylinders[0].id

34

# Ground points

In [43]:
out_dir = os.path.join(data_dir, 'thin_slice')
if not os.path.exists(out_dir):
    os.mkdir(out_dir)
# reference cylinder
c_ref = stations[0].refined_cylinders[2]
# a patch
ptc_fn = os.path.join(
    data_dir,
    f'{c_ref.station.name}-output',
    c_ref.ptc_filepath.split('\\')[-1]
)
c_ref = Cylinder(
    radius = c_ref.radius,
    position = c_ref.axis.position,
    vector = c_ref.axis.vector,
    metadata = {
        'ptc_filepath': ptc_fn,
        'station': c_ref.station,
        'id': c_ref.id
    }
)
c_ref.read_points()
c_ref.estimate_top_surface()
center_ref = c_ref.top_surface['center']
# center_ref = c_ref.axis.intersect_xy()
for s in stations:
    # if s.name != '057':
    #     continue
    try:
        cand_cyls = [c for c in s.refined_cylinders if c.radius > 9 and c.radius < 11.5]
        # print(len(cand_r))
        if len(cand_cyls) > 1:
            print(cand_cyls)
            # get the one that whose radius is closer to 10.6
            rs = [abs(c.radius-10.67)  for c in cand_cyls]
            i = rs.index(min(rs))
            # i = int(input('specify target: '))
            target_cyl = cand_cyls[i]
        else:
            target_cyl = cand_cyls[0]
        # a patch
        ptc_fn = os.path.join(
            data_dir,
            f'{target_cyl.station.name}-output',
            target_cyl.ptc_filepath.split('\\')[-1]
        )
        ori_target_cyl = deepcopy(target_cyl)

        target_cyl = Cylinder(
            radius = target_cyl.radius,
            position = target_cyl.axis.position,
            vector = target_cyl.axis.vector,
            metadata = {
                'ptc_filepath': ptc_fn,
                'station': target_cyl.station,
                'id': target_cyl.id
            }
        )
        target_cyl.read_points()
        # center
        target_cyl.estimate_top_surface()
        center = target_cyl.top_surface['center']
        # save
        s.target_cylinder = target_cyl
        
        # center = target_cyl.axis.intersect_xy()
        # translation
        t = center_ref - center
        target_cyl.match = [{
            'cylinder_ref': c_ref,
            'translation': t
        }]
        
        # 5m from the ground
        target_cyl.update_filepath(data_dir, 2)
        target_cyl.read_points()
        z_min = target_cyl.points[:,2].min() + 1
        z_max = z_min + 0.2

        ground_ptc = (target_cyl.points[:,2] >= z_min) & (target_cyl.points[:,2] < z_max)
        target_cyl.points = target_cyl.points[ground_ptc,:]

        # # 4m + radius
        # read points from related station
        # p = Path(target_cyl.station.file_path)

        # _s = Station(
        #     file_path = os.path.join(
        #         data_dir,
        #         p.name
        #     )
        # )

        # ground_ptc = (_s.ptc[:,2] >= z_min) & (_s.ptc[:,2] < z_max)
        # target_cyl.points = _s.ptc[ground_ptc,:]

        # # extend points
        # rho, theta = cart2pol(
        #     target_cyl.points[:,0] - center[0], 
        #     target_cyl.points[:,1] - center[1]
        # )

        # mask = rho <= 4.0 + target_cyl.radius
        # target_cyl.points = target_cyl.points[mask,:]

        # add big clusters in surrounding environment
        for cyl in s.refined_cylinders:
            if ori_target_cyl == cyl:
                print('skip')
                continue
            # fix file path
            cyl.update_filepath(data_dir, 2)
            cyl.read_points()
            ground_ptc = (cyl.points[:,2] >= z_min) & (cyl.points[:,2] < z_max)
            cyl.points = cyl.points[ground_ptc,:]
            target_cyl.points = np.r_[target_cyl.points, cyl.points]

        # keep xy only
        # target_cyl.points[:,2] = 0

        # translate and save
        t = target_cyl.match[0]['translation'].reshape([1,3])
        # t[2] = 0
        write_array2txt(
            os.path.join(out_dir, f'{s.name}.txt'),
            target_cyl.points + t
        )
        # write_array2txt(
        #     os.path.join(out_dir, f'{s.name}.txt'),
        #     points + t
            # cyl.ptc + t
        # )
    except Exception as exc:
        print(f'Error in {s}')
        traceback.print_exc()


top grid 25
polar grid cell: 65
[{'station': {'name': '055'}, 'radius': 10.670726, 'axis': "{'position': array([32.022026  , 13.261226  ,  0.04727127], dtype=float32), 'vector': array([-1.1702334e-03, -5.0637027e-04,  9.9999917e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\055-output\\055 cluster_7.txt'}, {'station': {'name': '055'}, 'radius': 10.611485, 'axis': "{'position': array([106.75986 ,  -8.452194,   8.3467  ], dtype=float32), 'vector': array([-0.06842601,  0.11503209,  0.99100226], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\055-output\\055 cluster_24.txt'}]
top grid 98
top grid 98
polar grid cell: 62
top grid 98
polar grid cell: 61
top grid 98
polar grid cell: 59
top grid 98
top grid 98
polar grid cell: 64
top grid 98
polar grid cell: 59
top grid 98
top grid 98
polar grid cell: 59
top grid 98
top grid 98
polar grid cell: 62
top grid 98
top grid 98
polar grid cell: 58
top grid 98
top grid 98


In [35]:
os.path.split('D:\\REGISTRATION DATA\\JB-483 70ft EXT\\055-output')

('D:\\REGISTRATION DATA\\JB-483 70ft EXT', '055-output')

In [39]:
target_cyl.update_filepath('afc', 3)

In [40]:
target_cyl.ptc_filepath

'afc\\JB-483 70ft EXT\\067-output\\067 cluster_7.txt'

In [33]:
os.path.normcase('D:\\REGISTRATION DATA\\JB-483 70ft EXT\\055-output\\055 cluster_24.txt')

'd:\\registration data\\jb-483 70ft ext\\055-output\\055 cluster_24.txt'

In [16]:
target_cyl.match[0]['translation']

array([ 17.60459978,  65.24013373, -13.68019956])

In [56]:
out_dir = os.path.join(data_dir, 'top_center_aligned_with_radius_extended')
if not os.path.exists(out_dir):
    os.mkdir(out_dir)
for s in stations:
    # if (s.name != '055') and (s.name != '056'):
    #     continue
    try:
        print(s)

        # read station point cloud
        station = Station(
            name='s.name', 
            file_path= os.path.join(
                data_dir,
                f'{s.name}.txt'
            )
        )
        # center of the target cylinder
        center = all_centers[s.name]
        print(center)
        rho, theta = cart2pol(
            station.ptc[:,0] - center[0,0], 
            station.ptc[:,1] - center[0,1]
        )

        # 1m + radius
        mask = rho <= 4.0 + all_radius[s.name]
        station.ptc = station.ptc[mask,:]
        
        for cyl in s.refined_cylinders:
            file_parts = cyl.ptc_filepath.split('\\')
            file_path = f'/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/{file_parts[-2]}/{file_parts[-1]}'
            if all_cyls[s.name] == file_path:
                print('skip')
                continue
            cyl = Station(
                file_path=file_path
            )
            station.ptc = np.r_[station.ptc, cyl.ptc]

        # save
        write_array2txt(
            os.path.join(out_dir, f'{s.name}.txt'),
            station.ptc + all_trans[s.name].reshape([1,3])
        )
    except Exception as exc:
        print(f'Error in {s}')
        traceback.print_exc()

{'name': '055'}
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483 055.txt
[[ 31.35794387  12.96720472 574.16282128]]
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output 055 cluster_2.txt
skip
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output 055 cluster_9.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output 055 cluster_24.txt
{'name': '056'}
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483 056.txt
[[ 19.93567675  26.22251586 573.44826667]]
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/056-output 056 cluster_11.txt
skip
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/056-output 056 cluster_15.txt


In [54]:
station.ptc.shape

(406609, 3)

In [112]:
write_array2txt(
    os.path.join(out_dir, 'top_057_all_high_points.txt'),
    # target_cyl.top_surface['center'].
    # target_cyl.points[target_cyl.all_top_ptc_ids,:]
    # target_cyl.points[target_cyl.shell_grids['ptc_ids'][99], :]
    # all_centers
    points
)